# Electricity Consumption Forecasting with a Simple RNN

A clean, leakage-aware portfolio notebook. The original notebook is preserved under `notebooks/archive/`.

## 1. Setup

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.data_preprocessing import load_csv, prepare_time_series
from src.feature_engineering import add_calendar_features
from src.sequence_generation import chronological_boundaries, create_partitioned_sequences

print(PROJECT_ROOT)

## 2. Load and clean the hourly demonstration data

The repository sample is explicitly synthetic and safe to publish. Replace it with the documented UCI Tetouan dataset for the real-data version.

In [ ]:
raw = load_csv(PROJECT_ROOT / "data" / "sample_input.csv")
cleaned, quality_report = prepare_time_series(raw, "timestamp", "consumption_kwh", frequency="h")
quality_report.to_dict(), cleaned.head()

## 3. Calendar features and chronological split

In [ ]:
featured = add_calendar_features(cleaned)
boundaries = chronological_boundaries(len(featured), TRAIN_RATIO, VALIDATION_RATIO)
boundaries

## 4. Train the genuine Keras SimpleRNN

Run the project training entrypoint so the notebook and command-line workflow stay consistent.

In [ ]:
# From the project root:
# !python train_model.py --data data/sample_input.csv --lookback 24 --epochs 60

## 5. Review metrics and baselines

In [ ]:
import json
import pandas as pd

metrics = json.loads((PROJECT_ROOT / "outputs" / "model_metrics.json").read_text())
comparison = pd.read_csv(PROJECT_ROOT / "outputs" / "baseline_comparison.csv")
metrics, comparison

## 6. Inspect saved forecasts and diagnostics

In [ ]:
from IPython.display import Image, display
for filename in ["consumption_trend.png", "training_curve.png", "actual_vs_predicted.png", "residual_plot.png", "forecast_plot.png"]:
    display(Image(filename=str(PROJECT_ROOT / "outputs" / filename)))

## 7. Load the reusable forecast pipeline

In [ ]:
from src.forecasting_pipeline import ElectricityForecastingPipeline

pipeline = ElectricityForecastingPipeline(PROJECT_ROOT / "models")
forecast = pipeline.forecast(cleaned[["timestamp", "consumption_kwh"]], horizon=24)
forecast.head(), pipeline.interpret(cleaned, forecast)